# Notebook 05 — Diagnóstico e seleção inicial dos modelos de sentimento para inadimplência (H=1)

## Objetivo do notebook

Este notebook tem como objetivo preparar a base consolidada que será consumida no Notebook 06 e realizar uma seleção diagnóstica dos modelos de sentimento que serão avaliados na etapa preditiva.

A lógica aplicada é:

1. carregar a base autorregressiva de inadimplência exportada pelo Notebook 02, agora com horizonte **H=1**;
2. carregar as séries de sentimento geradas no Notebook 04 para os modelos BERT, Mistral, TextBlob, NLTK/VADER e FinBERT;
3. criar um sexto indicador, chamado **Média dos Modelos**, calculado como a média dos scores de sentimento disponíveis;
4. mensalizar as séries de sentimento por tipo de documento (`copom` e `estatisticas`), mantendo uma grade mensal completa;
5. construir, para cada série de sentimento, a variável contemporânea ao mês de referência (`_t`) e as variáveis defasadas de 1 a 6 meses (`_L1` a `_L6`), e unir essa base à base de inadimplência;
6. exportar a base consolidada como `base_completa.csv` para consumo no Notebook 06;


> **Sobre as variáveis-alvo:** no Notebook 02, a coluna `data_ref` representa o mês de referência `t`, a coluna `data_alvo` representa o mês previsto (`data_ref + 1 mês`) e `inad_total_h1` representa a inadimplência observada em `t+1`.
>
> **Sobre os lags de sentimento:** a base exportada passa a preservar o sentimento contemporâneo ao mês de referência (`_t`) e as defasagens históricas (`_L1` a `_L6`). Assim, o Notebook 06 consegue montar janelas como `t`, `t-1` e `t-2` para cada série de sentimento, mantendo comparabilidade com a janela histórica da inadimplência usada nos modelos.
>


In [17]:
!pip install seaborn -q


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


## 1. Carregamento das bases

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Parâmetros centrais do Notebook 05 ──────────────────────────────────────
# H=1 significa que a inadimplência-alvo é observada um mês à frente.
HORIZONTE = 1
TARGET_COL = f"inad_total_h{HORIZONTE}"
DATA_REF_COL = "data_ref"
DATA_ALVO_COL = "data_alvo"

ARQUIVO_INAD = "base_modelagem_inadimplencia.csv"
ARQUIVO_SENT = "base_sentimentos.csv"

# ── Série de inadimplência — saída do Notebook 02 ───────────────────────────
# A base atual do Notebook 02 usa data_ref como mês de referência e data_alvo
# como mês previsto.
df_inad = pd.read_csv(ARQUIVO_INAD)
df_inad.columns = [c.strip().lower() for c in df_inad.columns]
df_inad = df_inad.drop(columns=[c for c in df_inad.columns if c.startswith("unnamed")], errors="ignore")

print(f"{ARQUIVO_INAD} carregado.")
print(f"Colunas da base de inadimplência: {list(df_inad.columns)}")

# ── Scores de sentimento — saída do Notebook 04 unificado ───────────────────
URL_SENT = "https://raw.githubusercontent.com/akemitti/Pred-inad-credito/main/base_sentimentos.csv"
try:
    df_sent = pd.read_csv(ARQUIVO_SENT)
    print(f"{ARQUIVO_SENT} carregado do disco.")
except FileNotFoundError:
    df_sent = pd.read_csv(URL_SENT)
    print(f"{ARQUIVO_SENT} carregado do GitHub.")

df_sent.columns = [c.strip().lower() for c in df_sent.columns]
df_sent["data"] = pd.to_datetime(df_sent["data"], errors="coerce")

# Identificar colunas de score exportadas pelo Notebook 04.
score_cols_raw = [c for c in df_sent.columns if c.startswith("score_")]
print(f"Scores disponíveis: {score_cols_raw}")


base_modelagem_inadimplencia.csv carregado.
Colunas da base de inadimplência: ['data_ref', 'inad_total', 'inad_total_l1', 'inad_total_l2', 'inad_total_l3', 'inad_total_l4', 'inad_total_l5', 'inad_total_l6', 'data_alvo', 'inad_total_h1']
base_sentimentos.csv carregado do disco.
Scores disponíveis: ['score_nltk', 'score_textblob', 'score_bert', 'score_finbert', 'score_mistral']


In [19]:
df_inad

,data_ref,inad_total,inad_total_l1,inad_total_l2,inad_total_l3,inad_total_l4,inad_total_l5,inad_total_l6,data_alvo,inad_total_h1
0,2019-07-01,3.06,2.95,3.05,3.02,2.99,2.91,2.95,2019-08-01,3.04
1,2019-08-01,3.04,3.06,2.95,3.05,3.02,2.99,2.91,2019-09-01,3.06
2,2019-09-01,3.06,3.04,3.06,2.95,3.05,3.02,2.99,2019-10-01,3.03
3,2019-10-01,3.03,3.06,3.04,3.06,2.95,3.05,3.02,2019-11-01,3.00
4,2019-11-01,3.00,3.03,3.06,3.04,3.06,2.95,3.05,2019-12-01,2.94
...,...,...,...,...,...,...,...,...,...,...
72,2025-07-01,3.77,3.57,3.54,3.50,3.28,3.26,3.19,2025-08-01,3.95
73,2025-08-01,3.95,3.77,3.57,3.54,3.50,3.28,3.26,2025-09-01,3.91
74,2025-09-01,3.91,3.95,3.77,3.57,3.54,3.50,3.28,2025-10-01,4.00
75,2025-10-01,4.00,3.91,3.95,3.77,3.57,3.54,3.50,2025-11-01,4.05


In [20]:
df_sent

,data,tipo_relatorio,arquivo,tipo,score_nltk,score_textblob,score_bert,score_finbert,score_mistral
0,2019-02-06,copom,COPOM220-not20190206220.pdf,copom,-0.321211,0.030093,0.111111,-0.312608,0.005556
1,2019-03-20,copom,COPOM221-not20190320221.pdf,copom,-0.367144,-0.004187,-0.002412,-0.354887,-0.058400
2,2019-05-08,copom,COPOM222-not20190508222.pdf,copom,-0.474129,0.019940,-0.047856,-0.304247,0.166654
3,2019-06-19,copom,Copom223-not20190619223.pdf,copom,-0.368450,0.033056,0.050000,-0.292347,0.060000
4,2019-07-31,copom,Copom224-not20190731224.pdf,copom,-0.395428,0.029630,0.030835,-0.306389,0.325654
...,...,...,...,...,...,...,...,...,...
129,2025-08-01,estatisticas,202508_Texto_de_estatisticas_monetarias_e_de_c...,estatisticas,-0.833226,0.076864,-1.000000,0.083134,-0.172207
130,2025-09-01,estatisticas,202509_Texto_de_estatisticas_monetarias_e_de_c...,estatisticas,-0.738277,-0.067303,-0.816446,-0.013248,-0.297234
131,2025-10-01,estatisticas,202510_Texto_de_estatisticas_monetarias_e_de_c...,estatisticas,-0.833063,-0.169941,-1.000000,0.323249,0.187804
132,2025-11-01,estatisticas,202511_Texto_de_estatisticas_monetarias_e_de_c...,estatisticas,-0.861543,-0.125786,-0.921384,0.534591,-0.207547


## 2. Agregação mensal com grade completa e interpolação

A série de inadimplência é mensal. Como as **Atas do Copom** não ocorrem exatamente todos os meses — em geral, seguem o calendário de reuniões do Copom, que pode ocorrer em intervalos próximos de 45 dias — é necessário transformar o sentimento das atas em uma série mensal contínua.

A regra aplicada neste notebook é:

1. agregar os scores de sentimento por mês e por tipo de relatório;
2. criar uma grade mensal completa de `2019-01-01` até `2025-12-01`;
3. garantir que, para todos os meses, exista valor de sentimento para as **Atas do Copom**, usando interpolação linear para preencher lacunas internas;
4. aplicar `bfill/ffill` — preenchimento com o valor seguinte/anterior — apenas nas bordas da série, quando necessário;
5. manter colunas de rastreabilidade para identificar quais meses eram observados originalmente e quais foram preenchidos.

Os Relatórios de Estatísticas também são mantidos na base consolidada, mas o ponto metodológico mais importante aqui é a mensalização das Atas do Copom para compatibilização com a série mensal de inadimplência.

In [21]:
df_sent_completo = df_sent.copy()

# ── Padronização da identificação do tipo de relatório ──────────────────────
if "tipo_relatorio" not in df_sent_completo.columns:
    if "tipo" in df_sent_completo.columns:
        df_sent_completo["tipo_relatorio"] = df_sent_completo["tipo"]
    else:
        raise ValueError(
            "A coluna 'tipo_relatorio' não foi encontrada na base_sentimentos.csv. "
            "Verifique se o Notebook 04 exportou a base consolidada corretamente."
        )

df_sent_completo["tipo_relatorio"] = (
    df_sent_completo["tipo_relatorio"]
    .astype(str)
    .str.lower()
    .str.strip()
)

TIPOS_VALIDOS = ["copom", "estatisticas"]
df_sent_completo = df_sent_completo[
    df_sent_completo["tipo_relatorio"].isin(TIPOS_VALIDOS)
].copy()

# ── Padronização da data e dos scores ───────────────────────────────────────
df_sent_completo["data"] = pd.to_datetime(df_sent_completo["data"], errors="coerce")
df_sent_completo = df_sent_completo.dropna(subset=["data"])

# Garantir que os scores sejam numéricos
for col in score_cols_raw:
    df_sent_completo[col] = pd.to_numeric(df_sent_completo[col], errors="coerce")

score_cols = [c for c in score_cols_raw if df_sent_completo[c].notna().any()]

print("Scores válidos para análise:")
print(score_cols)

print("\nDistribuição da base completa por tipo de relatório:")
display(
    df_sent_completo["tipo_relatorio"]
    .value_counts()
    .rename_axis("tipo_relatorio")
    .reset_index(name="qtd_registros")
)

# ── Grade mensal completa ───────────────────────────────────────────────────
PERIODO_INICIO = "2019-01-01"
PERIODO_FIM = "2025-12-01"

grade_mensal = pd.date_range(
    start=PERIODO_INICIO,
    end=PERIODO_FIM,
    freq="MS"
)

print(f"\nGrade mensal esperada: {len(grade_mensal)} meses")
print(f"Período: {grade_mensal.min().date()} a {grade_mensal.max().date()}")


def agregar_mensal_com_interpolacao(df_base, tipo_relatorio, score_cols, grade_mensal):
    """
    Agrega os scores por mês para um tipo de relatório e garante uma série mensal completa.

    Para as Atas do Copom, essa etapa é essencial porque as atas não são mensais.
    A interpolação linear preenche lacunas internas entre observações conhecidas.
    """
    df_tipo = df_base[df_base["tipo_relatorio"] == tipo_relatorio].copy()

    if df_tipo.empty:
        raise ValueError(f"Não há registros para tipo_relatorio = '{tipo_relatorio}'.")

    df_tipo["mes"] = df_tipo["data"].dt.to_period("M").dt.to_timestamp()

    # Agregação mensal: se houver mais de um documento no mesmo mês, usa a média.
    df_obs = (
        df_tipo.groupby("mes")[score_cols]
        .mean()
        .sort_index()
    )

    meses_observados = set(df_obs.index)

    # Grade mensal completa
    df_mensal = df_obs.reindex(grade_mensal)
    df_mensal.index.name = "data"
    df_mensal = df_mensal.reset_index()

    df_mensal["tipo_relatorio"] = tipo_relatorio
    df_mensal["registro_original"] = df_mensal["data"].isin(meses_observados)
    df_mensal["mes_ausente_originalmente"] = ~df_mensal["registro_original"]

    # Preenchimento das lacunas internas
    scores_interpolados = df_mensal[score_cols].interpolate(
        method="linear",
        limit_area="inside"
    )

    # Preenchimento de bordas, quando necessário
    scores_preenchidos = scores_interpolados.bfill().ffill()

    df_mensal[score_cols] = scores_preenchidos

    # Rastreabilidade do método de preenchimento
    df_mensal["metodo_preenchimento"] = "observado"

    mask_interpolado = (
        df_mensal["mes_ausente_originalmente"]
        & scores_interpolados.notna().any(axis=1)
    )
    df_mensal.loc[mask_interpolado, "metodo_preenchimento"] = "interpolacao_linear"

    mask_borda = (
        df_mensal["mes_ausente_originalmente"]
        & ~mask_interpolado
    )
    df_mensal.loc[mask_borda, "metodo_preenchimento"] = "bfill_ffill_borda"

    return df_mensal


# Gerar base mensal completa para cada tipo de relatório
dfs_mensais = []
for tipo in TIPOS_VALIDOS:
    df_tipo_mensal = agregar_mensal_com_interpolacao(
        df_base=df_sent_completo,
        tipo_relatorio=tipo,
        score_cols=score_cols,
        grade_mensal=grade_mensal
    )
    dfs_mensais.append(df_tipo_mensal)

df_sent_mensal = pd.concat(dfs_mensais, ignore_index=True)

# Verificação da cobertura mensal
resumo_cobertura = (
    df_sent_mensal
    .groupby("tipo_relatorio")
    .agg(
        meses_total=("data", "nunique"),
        registros_originais=("registro_original", "sum"),
        meses_preenchidos=("mes_ausente_originalmente", "sum")
    )
    .reset_index()
)

print("\n=== Cobertura mensal após interpolação ===")
display(resumo_cobertura)

print("\nMétodo de preenchimento por tipo de relatório:")
display(
    df_sent_mensal
    .groupby(["tipo_relatorio", "metodo_preenchimento"])
    .size()
    .reset_index(name="qtd_meses")
)

# Checagem específica: Copom deve ter todos os meses.
qtd_meses_copom = df_sent_mensal.query("tipo_relatorio == 'copom'")["data"].nunique()
assert qtd_meses_copom == len(grade_mensal), (
    f"Copom deveria ter {len(grade_mensal)} meses, mas tem {qtd_meses_copom}."
)

print(f"\n✅ Série mensal do Copom completa: {qtd_meses_copom} meses.")
display(df_sent_mensal.head())

Scores válidos para análise:
['score_nltk', 'score_textblob', 'score_bert', 'score_finbert', 'score_mistral']

Distribuição da base completa por tipo de relatório:


,tipo_relatorio,qtd_registros
0,estatisticas,81
1,copom,53



Grade mensal esperada: 84 meses
Período: 2019-01-01 a 2025-12-01

=== Cobertura mensal após interpolação ===


,tipo_relatorio,meses_total,registros_originais,meses_preenchidos
0,copom,84,53,31
1,estatisticas,84,81,3



Método de preenchimento por tipo de relatório:


,tipo_relatorio,metodo_preenchimento,qtd_meses
0,copom,bfill_ffill_borda,1
1,copom,interpolacao_linear,30
2,copom,observado,53
3,estatisticas,interpolacao_linear,3
4,estatisticas,observado,81



✅ Série mensal do Copom completa: 84 meses.


,data,score_nltk,score_textblob,score_bert,score_finbert,score_mistral,tipo_relatorio,registro_original,mes_ausente_originalmente,metodo_preenchimento
0,2019-01-01,-0.321211,0.030093,0.111111,-0.312608,0.005556,copom,False,True,bfill_ffill_borda
1,2019-02-01,-0.321211,0.030093,0.111111,-0.312608,0.005556,copom,True,False,observado
2,2019-03-01,-0.367144,-0.004187,-0.002412,-0.354887,-0.058400,copom,True,False,observado
3,2019-04-01,-0.420637,0.007877,-0.025134,-0.329567,0.054127,copom,False,True,interpolacao_linear
4,2019-05-01,-0.474129,0.019940,-0.047856,-0.304247,0.166654,copom,True,False,observado


## 3. Modelo 6 — Média dos modelos de sentimento

In [22]:
# Sentimento médio dos modelos válidos
df_sent_mensal["score_media"] = df_sent_mensal[score_cols].mean(axis=1)
score_cols_todos = score_cols + ["score_media"]

print("Score médio (primeiros 5):")
df_sent_mensal



Score médio (primeiros 5):


,data,score_nltk,score_textblob,score_bert,score_finbert,score_mistral,tipo_relatorio,registro_original,mes_ausente_originalmente,metodo_preenchimento,score_media
0,2019-01-01,-0.321211,0.030093,0.111111,-0.312608,0.005556,copom,False,True,bfill_ffill_borda,-0.097412
1,2019-02-01,-0.321211,0.030093,0.111111,-0.312608,0.005556,copom,True,False,observado,-0.097412
2,2019-03-01,-0.367144,-0.004187,-0.002412,-0.354887,-0.058400,copom,True,False,observado,-0.157406
3,2019-04-01,-0.420637,0.007877,-0.025134,-0.329567,0.054127,copom,False,True,interpolacao_linear,-0.142667
4,2019-05-01,-0.474129,0.019940,-0.047856,-0.304247,0.166654,copom,True,False,observado,-0.127928
...,...,...,...,...,...,...,...,...,...,...,...
163,2025-08-01,-0.833226,0.076864,-1.000000,0.083134,-0.172207,estatisticas,True,False,observado,-0.369087
164,2025-09-01,-0.738277,-0.067303,-0.816446,-0.013248,-0.297234,estatisticas,True,False,observado,-0.386502
165,2025-10-01,-0.833063,-0.169941,-1.000000,0.323249,0.187804,estatisticas,True,False,observado,-0.298390
166,2025-11-01,-0.861543,-0.125786,-0.921384,0.534591,-0.207547,estatisticas,True,False,observado,-0.316334


## 4. Preparação das bases para geração dos lags e merge

Nesta etapa, a base de inadimplência é padronizada para refletir a nova estrutura do Notebook 02.

A base atual deve conter:

- `data_ref`: mês de referência `t`, isto é, o mês cujas informações estão disponíveis no momento da previsão;
- `data_alvo`: mês previsto, igual a `data_ref + 1 mês`;
- `inad_total_h1`: inadimplência observada em `t+1`;
- `inad_total` e seus lags (`inad_total_L1` a `inad_total_L6`), que já foram criados no Notebook 02.

O Notebook 05 não recria os lags da inadimplência. Ele apenas acrescenta os lags de sentimento e exporta a base integrada para o Notebook 06.

In [23]:
# Cópias para evitar alterar os dataframes originais diretamente
# dataframe = estrutura tabular do pandas, parecida com uma planilha
df_inad_merge = df_inad.copy()
df_sent_lag_base = df_sent_mensal.copy()

# ── Parâmetro principal da etapa ────────────────────────────────────────────
# Os lags de sentimento devem ser calculados a partir de 2019-01-01,
# antes do merge com a base de inadimplência.
DATA_INICIO_LAGS_SENTIMENTO = pd.Timestamp("2019-01-01")

# ── Preparação da base de inadimplência ─────────────────────────────────────
# A base de inadimplência já vem do Notebook 02 com data_ref, data_alvo,
# inad_total_h1 e os lags da própria série. Esses lags NÃO são recriados aqui.
df_inad_merge.columns = [c.strip().lower() for c in df_inad_merge.columns]
df_inad_merge = df_inad_merge.drop(
    columns=[c for c in df_inad_merge.columns if c.startswith("unnamed")],
    errors="ignore"
)

# Compatibilidade com versões antigas: se ainda existir uma coluna chamada data,
# ela é interpretada como data_ref.
if DATA_REF_COL not in df_inad_merge.columns:
    if "data" in df_inad_merge.columns:
        df_inad_merge = df_inad_merge.rename(columns={"data": DATA_REF_COL})
        print("Coluna 'data' renomeada para 'data_ref' para compatibilidade.")
    else:
        raise ValueError(
            "A base de inadimplência precisa ter 'data_ref' ou 'data'. "
            "Verifique a exportação do Notebook 02."
        )

if TARGET_COL not in df_inad_merge.columns:
    raise ValueError(
        f"Coluna-alvo {TARGET_COL} não encontrada. "
        "Verifique se o Notebook 02 foi executado com HORIZONTE = 1."
    )

# data_alvo é obrigatória para os notebooks seguintes. Caso uma versão antiga
# não a traga, ela é reconstruída a partir de data_ref + HORIZONTE.
df_inad_merge[DATA_REF_COL] = pd.to_datetime(df_inad_merge[DATA_REF_COL], errors="coerce")

if DATA_ALVO_COL not in df_inad_merge.columns:
    df_inad_merge[DATA_ALVO_COL] = df_inad_merge[DATA_REF_COL] + pd.DateOffset(months=HORIZONTE)
    print("Coluna data_alvo criada a partir de data_ref + 1 mês.")
else:
    df_inad_merge[DATA_ALVO_COL] = pd.to_datetime(df_inad_merge[DATA_ALVO_COL], errors="coerce")

# A coluna mes será usada somente como chave técnica de merge.
# Ela corresponde ao mês de referência t.
df_inad_merge["mes"] = df_inad_merge[DATA_REF_COL].dt.to_period("M").dt.to_timestamp()

# Checagem da consistência do horizonte H=1.
mask_datas_validas = df_inad_merge[[DATA_REF_COL, DATA_ALVO_COL]].notna().all(axis=1)
data_alvo_esperada = df_inad_merge.loc[mask_datas_validas, DATA_REF_COL] + pd.DateOffset(months=HORIZONTE)
qtd_datas_inconsistentes = (
    df_inad_merge.loc[mask_datas_validas, DATA_ALVO_COL].dt.to_period("M").dt.to_timestamp()
    != data_alvo_esperada.dt.to_period("M").dt.to_timestamp()
).sum()

if qtd_datas_inconsistentes > 0:
    print(
        f"Atenção: {qtd_datas_inconsistentes} linha(s) têm data_alvo diferente de data_ref + {HORIZONTE} mês."
    )
else:
    print("✅ data_alvo compatível com H=1: data_alvo = data_ref + 1 mês.")

df_inad_merge = (
    df_inad_merge
    .dropna(subset=["mes", DATA_ALVO_COL, TARGET_COL])
    .drop_duplicates(subset=["mes"])
    .sort_values("mes")
    .reset_index(drop=True)
)

# ── Preparação da base mensal de sentimento ─────────────────────────────────
df_sent_lag_base["data"] = pd.to_datetime(df_sent_lag_base["data"], errors="coerce")
df_sent_lag_base["mes"] = df_sent_lag_base["data"].dt.to_period("M").dt.to_timestamp()

df_sent_lag_base = (
    df_sent_lag_base
    .dropna(subset=["mes", "tipo_relatorio"])
    .query("mes >= @DATA_INICIO_LAGS_SENTIMENTO")
    .sort_values(["tipo_relatorio", "mes"])
    .reset_index(drop=True)
)

# Conferir se a base mensal de sentimento tem apenas uma linha por mês e tipo de relatório.
# Isso é importante porque os lags precisam ser calculados em uma série temporal mensal única.
duplicados_sent = df_sent_lag_base[
    df_sent_lag_base.duplicated(subset=["mes", "tipo_relatorio"], keep=False)
]

if not duplicados_sent.empty:
    print("Atenção: foram encontrados meses duplicados na base mensal de sentimento.")
    display(duplicados_sent[["data", "mes", "tipo_relatorio"] + score_cols_todos].head(10))
else:
    print("✅ Base de sentimento mensal sem duplicidade por mês e tipo de relatório.")

print("=== Bases preparadas ===")
print(
    f"Inadimplência: {len(df_inad_merge)} linhas | "
    f"ref: {df_inad_merge[DATA_REF_COL].min().date()} a {df_inad_merge[DATA_REF_COL].max().date()} | "
    f"alvo: {df_inad_merge[DATA_ALVO_COL].min().date()} a {df_inad_merge[DATA_ALVO_COL].max().date()}"
)
print(
    f"Sentimento mensal para lags: {len(df_sent_lag_base)} linhas | "
    f"período: {df_sent_lag_base['mes'].min().date()} a {df_sent_lag_base['mes'].max().date()}"
)

print("\nCobertura da base de sentimento por tipo de relatório:")
display(
    df_sent_lag_base
    .groupby("tipo_relatorio")
    .agg(
        inicio=("mes", "min"),
        fim=("mes", "max"),
        qtd_meses=("mes", "nunique")
    )
    .reset_index()
)


✅ data_alvo compatível com H=1: data_alvo = data_ref + 1 mês.
✅ Base de sentimento mensal sem duplicidade por mês e tipo de relatório.
=== Bases preparadas ===
Inadimplência: 77 linhas | ref: 2019-07-01 a 2025-11-01 | alvo: 2019-08-01 a 2025-12-01
Sentimento mensal para lags: 168 linhas | período: 2019-01-01 a 2025-12-01

Cobertura da base de sentimento por tipo de relatório:


,tipo_relatorio,inicio,fim,qtd_meses
0,copom,2019-01-01,2025-12-01,84
1,estatisticas,2019-01-01,2025-12-01,84


## 5. Geração do sentimento contemporâneo e dos lags antes do merge

A base final exportada para o Notebook 06 deve conter, para cada modelo de sentimento, tanto o valor contemporâneo ao mês de referência quanto as defasagens de 1 até 6 meses.

Essa escolha não é arbitrária. A janela de sentimentos foi definida para manter paralelismo metodológico com a janela histórica da própria série de inadimplência usada no Notebook 02. Dessa forma, quando um modelo utiliza uma determinada janela temporal da inadimplência, o Notebook 06 também consegue avaliar a série de sentimento dentro de uma janela equivalente.

A convenção adotada é:

- `_t`: sentimento observado no próprio mês de referência `t`;
- `_L1`: sentimento defasado em 1 mês, isto é, observado em `t−1`;
- `_L2`: sentimento defasado em 2 meses, isto é, observado em `t−2`;
- `_L3`: sentimento defasado em 3 meses, isto é, observado em `t−3`;
- ...
- `_L6`: sentimento defasado em 6 meses, isto é, observado em `t−6`.

Exemplo de notação:

```text
copom_score_mistral_t
copom_score_mistral_L1
copom_score_mistral_L2
copom_score_mistral_L3
copom_score_mistral_L4
copom_score_mistral_L5
copom_score_mistral_L6
```

Os lags são calculados sobre a série mensal de sentimento iniciada em `2019-01-01`. Depois disso, a base de sentimento é limpa com `dropna()`, isto é, são removidas as linhas iniciais que ainda não possuem histórico suficiente para preencher toda a janela de defasagens. Só então ocorre o merge, ou seja, a junção entre a base de sentimento e a base de inadimplência.

Como o projeto passou a usar horizonte **H=1**, a relação temporal da base final é:

```text
sentimento em t, t−1, t−2, ..., t−6  →  inadimplência em t+1
```

Em outras palavras, para uma linha com mês de referência `t`, a variável-alvo `inad_total_h1` corresponde à inadimplência em `t+1`, enquanto `copom_score_mistral_t` corresponde ao sentimento observado em `t`, `copom_score_mistral_L1` corresponde ao sentimento observado em `t−1` e assim sucessivamente.

Essa estrutura permite que o Notebook 06 estime modelos com janelas de sentimento compatíveis com as janelas históricas usadas na inadimplência. Por exemplo, se um modelo usar uma janela de três períodos, a série de sentimento poderá entrar como `sentimento_t`, `sentimento_L1` e `sentimento_L2`, isto é, valores observados em `t`, `t−1` e `t−2`.


In [24]:
# Definição dos lags que serão exportados.
# A coluna _t também será exportada e representa o sentimento no próprio mês de referência.
LAGS_EXPORTACAO = list(range(1, 7))


def gerar_base_sentimento_por_tipo(df_sent_mensal, tipo_relatorio, score_cols, lags):
    """
    Gera uma base em formato wide, com uma linha por mês e colunas de sentimento
    contemporâneas e defasadas.

    wide = formato largo, isto é, variáveis em colunas.

    Convenção temporal:
    - sufixo _t  = sentimento observado no mês de referência t;
    - sufixo _L1 = sentimento observado em t-1;
    - sufixo _L2 = sentimento observado em t-2;
    - ...
    - sufixo _L6 = sentimento observado em t-6.

    Importante:
    os lags são calculados a partir da série completa de sentimento mensal,
    iniciada em 2019-01-01, antes do merge com a inadimplência.

    Assim, por exemplo, o L6 de 2019-07 consegue usar o sentimento de 2019-01,
    mesmo que a base de modelagem da inadimplência comece apenas em 2019-07.
    """

    df_tipo = (
        df_sent_mensal[df_sent_mensal["tipo_relatorio"] == tipo_relatorio]
        .copy()
    )

    if df_tipo.empty:
        raise ValueError(f"Não há registros para tipo_relatorio = '{tipo_relatorio}'.")

    df_tipo["data"] = pd.to_datetime(df_tipo["data"], errors="coerce")
    df_tipo["mes"] = df_tipo["data"].dt.to_period("M").dt.to_timestamp()

    df_tipo = (
        df_tipo
        .dropna(subset=["mes"])
        .query("mes >= @DATA_INICIO_LAGS_SENTIMENTO")
        .groupby("mes", as_index=False)[score_cols]
        .mean()
        .sort_values("mes")
        .reset_index(drop=True)
    )

    df_sent_wide = df_tipo[["mes"]].copy()
    colunas_sentimento = []

    for col in score_cols:
        # Sentimento contemporâneo ao mês de referência t.
        nome_coluna_t = f"{tipo_relatorio}_{col}_t"
        df_sent_wide[nome_coluna_t] = df_tipo[col]
        colunas_sentimento.append(nome_coluna_t)

        # Sentimentos defasados: t-1, t-2, ..., t-6.
        for lag in lags:
            nome_coluna_lag = f"{tipo_relatorio}_{col}_L{lag}"
            df_sent_wide[nome_coluna_lag] = df_tipo[col].shift(lag)
            colunas_sentimento.append(nome_coluna_lag)

    # Limpeza feita ANTES do merge com a inadimplência.
    # dropna = remoção das linhas que ainda não possuem histórico suficiente para toda a janela.
    df_sent_wide = (
        df_sent_wide
        .dropna(subset=colunas_sentimento)
        .sort_values("mes")
        .reset_index(drop=True)
    )

    return df_sent_wide


# ── 1) Gerar base de sentimento com valor em t e lags, limpa, antes do merge ─
df_sent_lags_limpa = None

for tipo in TIPOS_VALIDOS:
    df_tipo_lags = gerar_base_sentimento_por_tipo(
        df_sent_mensal=df_sent_lag_base,
        tipo_relatorio=tipo,
        score_cols=score_cols_todos,
        lags=LAGS_EXPORTACAO
    )

    if df_sent_lags_limpa is None:
        df_sent_lags_limpa = df_tipo_lags.copy()
    else:
        df_sent_lags_limpa = df_sent_lags_limpa.merge(
            df_tipo_lags,
            on="mes",
            how="outer",
            validate="one_to_one"
        )

# Garantir ordenação e limpar qualquer nulo residual nas colunas de sentimento.
colunas_sentimento_exportadas = [
    c for c in df_sent_lags_limpa.columns
    if c != "mes" and c.startswith(tuple(TIPOS_VALIDOS))
]

# Mantém o nome antigo como alias para compatibilidade com as células seguintes.
colunas_sentimento_lag = colunas_sentimento_exportadas

colunas_sentimento_t = [c for c in colunas_sentimento_exportadas if c.endswith("_t")]
colunas_sentimento_defasado = [c for c in colunas_sentimento_exportadas if "_L" in c]

df_sent_lags_limpa = (
    df_sent_lags_limpa
    .sort_values("mes")
    .dropna(subset=colunas_sentimento_exportadas)
    .reset_index(drop=True)
)

print("=== Base de sentimento com valor em t e lags, limpa antes do merge ===")
print(f"Linhas: {len(df_sent_lags_limpa)}")
print(f"Período: {df_sent_lags_limpa['mes'].min().date()} a {df_sent_lags_limpa['mes'].max().date()}")
print(f"Quantidade de colunas de sentimento em t: {len(colunas_sentimento_t)}")
print(f"Quantidade de colunas de sentimento defasado: {len(colunas_sentimento_defasado)}")
print(f"Quantidade total de colunas de sentimento exportadas: {len(colunas_sentimento_exportadas)}")

print("\nValores nulos nas colunas de sentimento após dropna():")
display(
    df_sent_lags_limpa[colunas_sentimento_exportadas]
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "coluna", 0: "qtd_nulos"})
    .query("qtd_nulos > 0")
)

# ── 2) Fazer o merge somente depois da limpeza dos lags de sentimento ───────
# A base de inadimplência já traz os lags da inadimplência criados no Notebook 02.
# Aqui são adicionados o sentimento em t e os lags de sentimento.
df_export_final = df_inad_merge.merge(
    df_sent_lags_limpa,
    on="mes",
    how="inner",
    validate="one_to_one"
)

# Manter uma coluna data como referência mensal para compatibilidade com notebooks anteriores.
# A coluna data_ref também é preservada para deixar explícito o mês de referência t.
df_export_final["data"] = df_export_final["mes"]
df_export_final[DATA_REF_COL] = pd.to_datetime(df_export_final[DATA_REF_COL], errors="coerce").dt.to_period("M").dt.to_timestamp()
df_export_final[DATA_ALVO_COL] = pd.to_datetime(df_export_final[DATA_ALVO_COL], errors="coerce").dt.to_period("M").dt.to_timestamp()

# Organizar as principais colunas no início e remover mes da exportação final.
cols_inicio = [
    "data",
    DATA_REF_COL,
    DATA_ALVO_COL,
    TARGET_COL,
    "inad_total",
]
cols_inicio = [c for c in cols_inicio if c in df_export_final.columns]

cols_ordenadas = cols_inicio + [
    c for c in df_export_final.columns
    if c not in cols_inicio + ["mes"]
]

df_export_final = df_export_final[cols_ordenadas]

# Checagens finais
assert df_export_final["data"].nunique() == len(df_export_final), (
    "A base final deveria ter apenas uma linha por mês. Verifique duplicidades."
)

nulos_sent_final = df_export_final[colunas_sentimento_exportadas].isna().sum().sum()
assert nulos_sent_final == 0, (
    "Ainda há valores nulos nas colunas de sentimento após o merge. "
    "Verifique a base df_sent_lags_limpa."
)

assert TARGET_COL in df_export_final.columns, f"A coluna-alvo {TARGET_COL} não está na base final."
assert df_export_final[TARGET_COL].isna().sum() == 0, f"Há valores nulos em {TARGET_COL}."

print("\n=== Base final em formato wide para o Notebook 06 ===")
print(f"Linhas: {len(df_export_final)}")
print(f"Período de referência: {df_export_final[DATA_REF_COL].min().date()} a {df_export_final[DATA_REF_COL].max().date()}")
print(f"Período previsto:      {df_export_final[DATA_ALVO_COL].min().date()} a {df_export_final[DATA_ALVO_COL].max().date()}")
print(f"Target: {TARGET_COL}")
print(f"Quantidade de colunas: {len(df_export_final.columns)}")

print("\nExemplo de colunas de sentimento em t criadas:")
print(colunas_sentimento_t[:12])

print("\nExemplo de colunas de sentimento defasadas criadas:")
print(colunas_sentimento_defasado[:12])

print("\nConferência do merge:")
print(f"Meses na base de inadimplência: {df_inad_merge['mes'].nunique()}")
print(f"Meses na base de sentimento com t e lags limpa: {df_sent_lags_limpa['mes'].nunique()}")
print(f"Meses após merge final: {df_export_final['data'].nunique()}")

display(df_export_final.head(8))


=== Base de sentimento com valor em t e lags, limpa antes do merge ===
Linhas: 78
Período: 2019-07-01 a 2025-12-01
Quantidade de colunas de sentimento em t: 12
Quantidade de colunas de sentimento defasado: 72
Quantidade total de colunas de sentimento exportadas: 84

Valores nulos nas colunas de sentimento após dropna():


,coluna,qtd_nulos



=== Base final em formato wide para o Notebook 06 ===
Linhas: 77
Período de referência: 2019-07-01 a 2025-11-01
Período previsto:      2019-08-01 a 2025-12-01
Target: inad_total_h1
Quantidade de colunas: 95

Exemplo de colunas de sentimento em t criadas:
['copom_score_nltk_t', 'copom_score_textblob_t', 'copom_score_bert_t', 'copom_score_finbert_t', 'copom_score_mistral_t', 'copom_score_media_t', 'estatisticas_score_nltk_t', 'estatisticas_score_textblob_t', 'estatisticas_score_bert_t', 'estatisticas_score_finbert_t', 'estatisticas_score_mistral_t', 'estatisticas_score_media_t']

Exemplo de colunas de sentimento defasadas criadas:
['copom_score_nltk_L1', 'copom_score_nltk_L2', 'copom_score_nltk_L3', 'copom_score_nltk_L4', 'copom_score_nltk_L5', 'copom_score_nltk_L6', 'copom_score_textblob_L1', 'copom_score_textblob_L2', 'copom_score_textblob_L3', 'copom_score_textblob_L4', 'copom_score_textblob_L5', 'copom_score_textblob_L6']

Conferência do merge:
Meses na base de inadimplência: 77
Mes

,data,data_ref,data_alvo,inad_total_h1,inad_total,inad_total_l1,inad_total_l2,inad_total_l3,inad_total_l4,inad_total_l5,...,estatisticas_score_mistral_L4,estatisticas_score_mistral_L5,estatisticas_score_mistral_L6,estatisticas_score_media_t,estatisticas_score_media_L1,estatisticas_score_media_L2,estatisticas_score_media_L3,estatisticas_score_media_L4,estatisticas_score_media_L5,estatisticas_score_media_L6
0,2019-07-01,2019-07-01,2019-08-01,3.04,3.06,2.95,3.05,3.02,2.99,2.91,...,-0.110806,-0.640000,0.289589,-0.235622,-0.339155,-0.427217,-0.340231,-0.399399,-0.504163,-0.306098
1,2019-08-01,2019-08-01,2019-09-01,3.06,3.04,3.06,2.95,3.05,3.02,2.99,...,0.025000,-0.110806,-0.640000,-0.496210,-0.235622,-0.339155,-0.427217,-0.340231,-0.399399,-0.504163
2,2019-09-01,2019-09-01,2019-10-01,3.03,3.06,3.04,3.06,2.95,3.05,3.02,...,-0.410969,0.025000,-0.110806,-0.366679,-0.496210,-0.235622,-0.339155,-0.427217,-0.340231,-0.399399
3,2019-10-01,2019-10-01,2019-11-01,3.00,3.03,3.06,3.04,3.06,2.95,3.05,...,0.112556,-0.410969,0.025000,-0.393281,-0.366679,-0.496210,-0.235622,-0.339155,-0.427217,-0.340231
4,2019-11-01,2019-11-01,2019-12-01,2.94,3.00,3.03,3.06,3.04,3.06,2.95,...,0.186940,0.112556,-0.410969,-0.517378,-0.393281,-0.366679,-0.496210,-0.235622,-0.339155,-0.427217
5,2019-12-01,2019-12-01,2020-01-01,3.00,2.94,3.00,3.03,3.06,3.04,3.06,...,-0.428230,0.186940,0.112556,-0.424091,-0.517378,-0.393281,-0.366679,-0.496210,-0.235622,-0.339155
6,2020-01-01,2020-01-01,2020-02-01,3.04,3.00,2.94,3.00,3.03,3.06,3.04,...,-0.274115,-0.428230,0.186940,-0.358958,-0.424091,-0.517378,-0.393281,-0.366679,-0.496210,-0.235622
7,2020-02-01,2020-02-01,2020-03-01,3.18,3.04,3.00,2.94,3.00,3.03,3.06,...,-0.237136,-0.274115,-0.428230,-0.411280,-0.358958,-0.424091,-0.517378,-0.393281,-0.366679,-0.496210


In [25]:
# Exportação principal para consumo no Notebook 06
# index=False evita criar uma coluna extra chamada "Unnamed: 0".
df_export_final.to_csv("base_completa.csv", index=False)

print("Arquivo exportado: base_completa.csv")
print(f"Horizonte da base exportada: H={HORIZONTE} | Target: {TARGET_COL}")


Arquivo exportado: base_completa.csv
Horizonte da base exportada: H=1 | Target: inad_total_h1
